# HI-Small Data Preparation — Temporal Split, Baseline Features & GFP Structural Features

Adapted from my LI-Small pipeline (`AML_GNN_GMA/Data_preparation_gfp.ipynb`) for the **HI-Small** dataset.
Runs on the `graph_feature_preprocessor` kernel (Python 3.9, torch + PyG installed).

## Pipeline

1. **Load & truncate** — keep only 2022-09-01 → 2022-09-10; define the temporal split boundaries
2. **Baseline edge features** — FX-corrected `Amount_Log`, payment-format OHE, time cyclicals, plus new evidence-based features (Same_Bank, Δt, structuring band, causal bank risk target encoding)
3. **Node features** — entity-type OHE (6)
4. **GFP structural features** — IBM SnapML `GraphFeaturePreprocessor` (Altman et al. Appendix D config)
5. **Assembly & saving** — baseline and GFP groups kept **explicitly separated** in `feature_meta.json`
6. **Temporal 60/20/20 split + normalization** — scalers fit on train only (no leakage)
7. **Graph construction** — cumulative PyG snapshots (`train/val/test_graph.pt`)

## Feature inventory

| Group | Cols | Content |
|---|---|---|
| Baseline (BASE_EDGE_COLS) | 20 | `Amount_Log`, hour/day cyclicals, `Is_Weekend`, `Is_Self_Loop`, payment-format OHE (7), `Same_Bank`, `Dt_Src_Log`, `Dt_Dst_Log`, `Struct_Band`, `Src_Bank_Risk`, `Dst_Bank_Risk` |
| GFP (GFP_FEAT_COLS) | 61 | scatter-gather / temp-cycle / lc-cycle histograms (3×3) + vertex stats (4 groups × 13) |
| Node (NODE_FEAT_COLS) | 6 | entity-type OHE |

**Dropped from the LI-Small pipeline** (measured irrelevant on HI-Small): `Is_ACH` (exact duplicate
of `PayFmt_ACH`), `Currency_Mismatch` (zero positives in its 1.4% share), `Bank_ID_Norm`
(arbitrary ordinal — replaced by the causal, train-fit bank risk encoding on the edges).

The two edge-feature groups are saved together in one `edge_features.csv` but their column
lists are stored separately in `feature_meta.json` (`BASE_EDGE_COLS` vs `GFP_FEAT_COLS`),
so model notebooks can train **baseline-only**, **GFP-only**, or **combined** variants.

## Outputs (all in `Data/`)

`edge_features.csv` (raw, pre-normalization) · `node_features.csv` · `feature_meta.json` ·
`standard_scaler.pkl` · `train_graph.pt` / `val_graph.pt` / `test_graph.pt` · `account_to_idx.pkl`

## 0. Setup

In [1]:
import gc
import json
import os
import pickle
import subprocess
import time
import warnings

import numpy as np
import pandas as pd
import torch
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data

warnings.filterwarnings('ignore')

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self): self.t = time.time()
    def __exit__(self, *a): print(f'  [{self.label}] done in {time.time()-self.t:.1f}s')

print(f'numpy {np.__version__} | torch {torch.__version__} | CPUs: {os.cpu_count()}')

numpy 1.26.4 | torch 2.8.0+cpu | CPUs: 16


## 1. Load Data & Truncate at September 10

Per the dataset creator (Erik Altman, [Kaggle discussion](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/discussion/427517)), the *Small* datasets contain only **10 days of real data (Sep 1–10, 2022)**. The rows after Sep 10 exist only so that laundering patterns started near the end can complete — those extra days contain **no legitimate transactions**.

In HI-Small this tail is 1,108 rows with a **59% laundering rate** — 600× the true rate. Keeping it would place an artificial laundering cluster at the very end of the timeline, exactly where the temporal test split lives, so we **drop everything from 2022-09-11 onward** (losing 655 of 5,177 positives; 4,522 remain).

In [2]:
col_names = [
    'Timestamp', 'src_bank', 'src_account',
    'dst_bank', 'dst_account',
    'Amount Received', 'Receiving Currency',
    'Amount Paid', 'Payment Currency',
    'Payment Format', 'label'
]

edges = pd.read_csv('Data/HI-Small_Trans.csv', names=col_names, header=0, low_memory=False)
accounts = pd.read_csv('Data/HI-Small_accounts.csv', low_memory=False)

edges['Timestamp'] = pd.to_datetime(edges['Timestamp'])
n_before, pos_before = len(edges), edges['label'].sum()

# Keep only the 10 "real" days and sort by time (required by GFP and the temporal split)
edges = edges[edges['Timestamp'] < '2022-09-11']
edges = edges.sort_values('Timestamp', kind='stable').reset_index(drop=True)

print(f'Transactions: {n_before:,} -> {len(edges):,} (dropped {n_before - len(edges):,} tail rows)')
print(f'Laundering:   {pos_before:,} -> {edges["label"].sum():,} ({edges["label"].mean():.4%})')
print(f'Date range:   {edges["Timestamp"].min()} -> {edges["Timestamp"].max()}')
print(f'Accounts:     {len(accounts):,}')

# Temporal 60/20/20 split boundaries (positional, data is time-sorted).
# Defined here because the bank risk encoding in Section 2 must know
# where the training window ends.
n_edges = len(edges)
t1 = int(n_edges * 0.60)   # end of train
t2 = int(n_edges * 0.80)   # end of val
print(f'\nSplit boundaries: train ends at row {t1:,} '
      f'({edges["Timestamp"].iloc[t1-1]}), val ends at row {t2:,} '
      f'({edges["Timestamp"].iloc[t2-1]})')

Transactions: 5,078,345 -> 5,077,237 (dropped 1,108 tail rows)
Laundering:   5,177 -> 4,522 (0.0891%)
Date range:   2022-09-01 00:00:00 -> 2022-09-10 23:59:00
Accounts:     518,581

Split boundaries: train ends at row 3,046,342 (2022-09-06 13:34:00), val ends at row 4,061,789 (2022-09-08 16:09:00)


## 2. Baseline Edge Features

`Amount_Log` uses **day-specific FX rates** from Yahoo Finance (weekends forward-filled; USD rows keep rate 1.0) so amounts are comparable across the 15 currencies — the raw maximum "amount" is a 1-trillion **Yen** row, meaningless without conversion.

Beyond the LI-Small baseline, four features were added because they show measurable signal on HI-Small (base laundering rate 0.089%):

| Feature | Evidence on HI-Small |
|---|---|
| `Same_Bank` | same-bank transactions: 0.012% laundering vs cross-bank 0.101% — 8× separation |
| `Dt_Src_Log` (time since sender's previous txn) | laundering senders fire in bursts: median gap 0.0h vs 0.3h |
| `Dt_Dst_Log` (time since receiver's previous txn) | laundering receivers are dormant mules: median gap 8.2h vs 0.4h |
| `Struct_Band` (USD amount in \$9–10k) | 0.275% laundering vs 0.088% — the classic structuring band, 3.1× lift |
| `Src/Dst_Bank_Risk` (bank target encoding) | 30k banks; among banks with ≥5k txns rates range 0% → 0.68% |

**Bank risk target encoding — leakage protection (3 layers):**
1. **Temporal isolation** — rates are fit on train rows only and *frozen*; val/test just look them up (unseen banks → global train prior), like a production model scoring new traffic.
2. **Causal expanding window for train rows** — a train edge at time *t* gets the rate of the bank's transactions with timestamp **< t** only, so its own label (and any future label) never enters its own feature — the same causality principle as GFP.
3. **Smoothing** (`m=200`) — small banks shrink toward the global rate, controlling the noisy-rate instability seen in the LI-Small v2 experiment.

Bank risk is kept on the **edges** (not the nodes) precisely because only a per-transaction feature can be made causal in this way; a static node attribute would have to include an account's own training labels.

Dropped vs the LI-Small pipeline: `Is_ACH` (duplicate of `PayFmt_ACH`), `Currency_Mismatch` (zero positives in its 1.4% share).

In [3]:
# Yahoo Finance tickers for the 14 non-USD currencies (HI-Small naming)
CURRENCY_TICKER = {
    'Australian Dollar': 'AUDUSD=X', 'Bitcoin':         'BTC-USD',
    'Brazil Real':       'BRLUSD=X', 'Canadian Dollar': 'CADUSD=X',
    'Euro':              'EURUSD=X', 'Mexican Peso':    'MXNUSD=X',
    'Ruble':             'RUBUSD=X', 'Rupee':           'INRUSD=X',
    'Saudi Riyal':       'SARUSD=X', 'Shekel':          'ILSUSD=X',
    'Swiss Franc':       'CHFUSD=X', 'UK Pound':        'GBPUSD=X',
    'Yen':               'JPYUSD=X', 'Yuan':            'CNYUSD=X',
}

dates = pd.date_range('2022-09-01', '2022-09-10')
fx_rate = {}  # (currency, 'YYYY-MM-DD') -> rate to USD

for currency, ticker in CURRENCY_TICKER.items():
    close = yf.download(ticker, start='2022-09-01', end='2022-09-12',
                        auto_adjust=True, progress=False)['Close']
    close = close.squeeze().reindex(dates).ffill().bfill()  # fill weekends
    for day, rate in close.items():
        fx_rate[(currency, str(day.date()))] = float(rate)
    print(f'{currency:18s} {ticker:9s} last close = {float(close.iloc[-1]):.4f}')

Australian Dollar  AUDUSD=X  last close = 0.6766


Bitcoin            BTC-USD   last close = 21680.5391


Brazil Real        BRLUSD=X  last close = 0.1918


Canadian Dollar    CADUSD=X  last close = 0.7645


Euro               EURUSD=X  last close = 1.0012


...

Shekel             ILSUSD=X  last close = 0.2912


Swiss Franc        CHFUSD=X  last close = 1.0324


UK Pound           GBPUSD=X  last close = 1.1522


Yen                JPYUSD=X  last close = 0.0070


Yuan               CNYUSD=X  last close = 0.1438


In [4]:
# Amount in USD, then log1p to compress the heavy tail
edges['Date'] = edges['Timestamp'].dt.date.astype(str)
edges['FX_Rate'] = [fx_rate.get((c, d), 1.0)          # USD rows fall back to 1.0
                    for c, d in zip(edges['Payment Currency'], edges['Date'])]
edges['Amount_USD'] = edges['Amount Paid'] * edges['FX_Rate']
edges['Amount_Log'] = np.log1p(edges['Amount_USD'])

# Structuring band: amounts placed just below the $10k reporting threshold
edges['Struct_Band'] = edges['Amount_USD'].between(9000, 10000).astype(np.int8)

# Cyclical time encodings + weekend flag
hour = edges['Timestamp'].dt.hour
dow  = edges['Timestamp'].dt.dayofweek
edges['Hour_Sin']      = np.sin(2 * np.pi * hour / 24)
edges['Hour_Cos']      = np.cos(2 * np.pi * hour / 24)
edges['DayOfWeek_Sin'] = np.sin(2 * np.pi * dow / 7)
edges['DayOfWeek_Cos'] = np.cos(2 * np.pi * dow / 7)
edges['Is_Weekend']    = (dow >= 5).astype(np.int8)

# Payment format one-hot + self-loop + same-bank flags
fmt_ohe = pd.get_dummies(edges['Payment Format'], prefix='PayFmt').astype(np.int8)
edges = pd.concat([edges, fmt_ohe], axis=1)
edges['Is_Self_Loop'] = (edges['src_account'] == edges['dst_account']).astype(np.int8)
edges['Same_Bank']    = (edges['src_bank'] == edges['dst_bank']).astype(np.int8)

# Time since the account's previous transaction (burst / dormant-mule signal).
# Causal by construction: each row only looks at its account's PAST rows.
# First-ever transactions have no gap -> filled with the full window (10 days).
seconds = edges['Timestamp'].astype('int64') // 10**9
TEN_DAYS = 10 * 24 * 3600
edges['Dt_Src_Log'] = np.log1p((seconds - seconds.groupby(edges['src_account']).shift())
                               .fillna(TEN_DAYS))
edges['Dt_Dst_Log'] = np.log1p((seconds - seconds.groupby(edges['dst_account']).shift())
                               .fillna(TEN_DAYS))

In [5]:
# Bank risk: smoothed target encoding with the 3-layer leakage protection
# described above (temporal isolation + causal expanding window + smoothing).
M = 200                                      # smoothing strength
is_train = np.arange(len(edges)) < t1
prior = edges.loc[is_train, 'label'].mean()  # global train laundering rate


def bank_risk(bank_col):
    bank, label = edges[bank_col], edges['label']

    # Train rows: rate over STRICTLY EARLIER transactions of the same bank
    # (cumsum/cumcount minus the current row -> the row's own label is excluded)
    past_pos = label.groupby(bank).cumsum() - label
    past_cnt = label.groupby(bank).cumcount()
    risk_causal = (past_pos + M * prior) / (past_cnt + M)

    # Val/test rows: rate frozen from the full train window
    train_stats = edges[is_train].groupby(bank_col)['label'].agg(['sum', 'count'])
    frozen = (train_stats['sum'] + M * prior) / (train_stats['count'] + M)
    risk_frozen = bank.map(frozen).fillna(prior)    # unseen bank -> prior

    return np.where(is_train, risk_causal, risk_frozen).astype(np.float32)


edges['Src_Bank_Risk'] = bank_risk('src_bank')
edges['Dst_Bank_Risk'] = bank_risk('dst_bank')

BASE_EDGE_COLS = (
    ['Amount_Log', 'Struct_Band',
     'Hour_Sin', 'Hour_Cos', 'DayOfWeek_Sin', 'DayOfWeek_Cos', 'Is_Weekend',
     'Is_Self_Loop', 'Same_Bank', 'Dt_Src_Log', 'Dt_Dst_Log',
     'Src_Bank_Risk', 'Dst_Bank_Risk']
    + list(fmt_ohe.columns)
)

print(f'Baseline edge features ({len(BASE_EDGE_COLS)}):')
print(BASE_EDGE_COLS)
print()
print(f'Bank risk sanity (train rows): prior={prior:.6f}  '
      f'src max={edges.loc[is_train, "Src_Bank_Risk"].max():.6f}  '
      f'dst max={edges.loc[is_train, "Dst_Bank_Risk"].max():.6f}')

Baseline edge features (20):
['Amount_Log', 'Struct_Band', 'Hour_Sin', 'Hour_Cos', 'DayOfWeek_Sin', 'DayOfWeek_Cos', 'Is_Weekend', 'Is_Self_Loop', 'Same_Bank', 'Dt_Src_Log', 'Dt_Dst_Log', 'Src_Bank_Risk', 'Dst_Bank_Risk', 'PayFmt_ACH', 'PayFmt_Bitcoin', 'PayFmt_Cash', 'PayFmt_Cheque', 'PayFmt_Credit Card', 'PayFmt_Reinvestment', 'PayFmt_Wire']

Bank risk sanity (train rows): prior=0.000754  src max=0.049814  dst max=0.020687


## 3. Node Features

Entity-type OHE — HI-Small has **6 entity types** (Partnership, Corporation, Sole, Country, Individual, Direct), one more than LI-Small's 5. The EDA showed Individual (0.139%) and Corporation (0.127%) carry above-average risk.

(`Bank_ID_Norm` from the LI-Small pipeline is dropped — a min-max-scaled arbitrary ID is a meaningless ordinal; bank identity enters through the causal edge-level risk encoding of Section 2.)

In [6]:
nodes = accounts.drop_duplicates(subset='Account Number').reset_index(drop=True)

# Entity type = first word of the entity name, e.g. "Corporation #33520" -> "Corporation"
nodes['Entity_Type'] = nodes['Entity Name'].str.extract(r'^([A-Za-z]+)', expand=False)
entity_ohe = pd.get_dummies(nodes['Entity_Type'], prefix='EntityType').astype(np.int8)
nodes = pd.concat([nodes, entity_ohe], axis=1)

NODE_FEAT_COLS = list(entity_ohe.columns)
node_features = nodes[['Account Number'] + NODE_FEAT_COLS].rename(
    columns={'Account Number': 'account_id'})

print(f'Node feature matrix: {node_features.shape}')
print(f'Features: {NODE_FEAT_COLS}')
print(f'\nEntity types:')
print(nodes['Entity_Type'].value_counts().to_string())

Node feature matrix: (518573, 7)
Features: ['EntityType_Corporation', 'EntityType_Country', 'EntityType_Direct', 'EntityType_Individual', 'EntityType_Partnership', 'EntityType_Sole']

Entity types:
Entity_Type
Partnership    189680
Corporation    172347
Sole           149047
Country          6692
Individual        740
Direct             67


## 4. Graph Feature Preprocessor (GFP) Structural Features

IBM SnapML's `GraphFeaturePreprocessor` computes the AML graph patterns of Altman et al. (Appendix D) for every transaction in a single pass.

**Causal correctness:** GFP processes edges in timestamp order and, for edge (u→v, t), uses only edges with timestamp < t — no leakage by construction.

| Pattern | Configuration | AML signal |
|---|---|---|
| Scatter-Gather | bins=[2,3,5], window=6h | Rapid gather-then-scatter |
| Temporal cycle | bins=[2,3,5], window=24h | Cycle-closing edges (round-trip → complex ring) |
| Length-constrained cycle | bins=[2,3,5], window=24h, max_len=6 | Simple cycles up to length 6 (paper uses 10; 6 = speed/memory compromise) |
| Vertex stats | source+dest × out/in, cols=[timestamp, Amount_USD], window=24h | fan, degree, ratio + avg/sum/var/skew/kurtosis per column |

Fan/degree histogram patterns are **disabled** — vertex stats already provide continuous fan/degree values, strictly more informative than coarse bins.

**Batched streaming (leakage fix vs my LI-Small pipeline):** calling `fit_transform` on the whole
dataset at once lets every edge see **all** other edges — verified experimentally: an account's
first-ever transaction received the account's full *future* out-degree (plus a double-insertion
artifact from `fit` + `transform`). The paper processes edges with **batch size 128** via streaming
`transform` calls; we do the same, so each edge only sees earlier batches + its own batch.
With current-edge-inclusive semantics, a first-ever transaction correctly gets fan = deg = 1.

> **Windows note:** the Windows build of `snapml` does not include the GFP native code, so the
> GFP step runs through **WSL** via `run_gfp_wsl.py` (venv `~/gfp_env` inside Ubuntu). The
> notebook saves the input matrix to `.npy`, calls the script, and loads the result back —
> numerically identical to running GFP directly (this is the same Linux build Kaggle uses).

In [7]:
GFP_PARAMS = {
    'num_threads': 8,
    'time_window': 24 * 60 * 60,

    # Vertex statistics over the input columns: 3 = timestamp, 4 = Amount_USD
    'vertex_stats':       True,
    'vertex_stats_tw':    24 * 60 * 60,
    'vertex_stats_cols':  [3, 4],
    'vertex_stats_feats': [0, 1, 2, 3, 4, 8, 9, 10],
    # feat ids: 0 fan, 1 degree, 2 ratio, 3 avg, 4 sum, 8 var, 9 skew, 10 kurtosis

    # fan/degree histograms disabled (vertex stats cover them better)
    'fan':    False, 'fan_bins':    [2, 5, 10],
    'degree': False, 'degree_bins': [2, 5, 10],

    'scatter-gather':      True,
    'scatter-gather_tw':   6 * 60 * 60,
    'scatter-gather_bins': [2, 3, 5],

    'temp-cycle':      True,
    'temp-cycle_tw':   24 * 60 * 60,
    'temp-cycle_bins': [2, 3, 5],

    'lc-cycle':      True,
    'lc-cycle_tw':   24 * 60 * 60,
    'lc-cycle_bins': [2, 3, 5],
    'lc-cycle_len':  6,
}

# Output column names, in the exact order GFP emits them:
# first the 3 pattern histograms (3 bins each) ...
GFP_FEAT_COLS = []
for pattern in ['scatter-gather', 'temp-cycle', 'lc-cycle']:
    GFP_FEAT_COLS += [f'{pattern}_bins_2-3', f'{pattern}_bins_3-5', f'{pattern}_bins_5-inf']

# ... then vertex stats: 4 groups (source/dest x out/in), each =
#     3 topology values + 5 stats for timestamps + 5 stats for amounts
STATS = ['avg', 'sum', 'var', 'skew', 'kurtosis']
for who in ['source', 'dest']:
    for direction in ['out', 'in']:
        GFP_FEAT_COLS += [f'{who}_fan_{direction}', f'{who}_deg_{direction}',
                          f'{who}_ratio_{direction}']
        for col in ['ts', 'amt']:
            GFP_FEAT_COLS += [f'{who}_{s}_{col}_{direction}' for s in STATS]

print(f'GFP feature columns: {len(GFP_FEAT_COLS)}  (9 pattern bins + 4 x 13 vertex stats)')
assert len(GFP_FEAT_COLS) == 61
GFP_FEAT_COLS

GFP feature columns: 61  (9 pattern bins + 4 x 13 vertex stats)


['scatter-gather_bins_2-3',
 'scatter-gather_bins_3-5',
 'scatter-gather_bins_5-inf',
 'temp-cycle_bins_2-3',
 'temp-cycle_bins_3-5',
 'temp-cycle_bins_5-inf',
 'lc-cycle_bins_2-3',
 'lc-cycle_bins_3-5',
 'lc-cycle_bins_5-inf',
 'source_fan_out',
 'source_deg_out',
 'source_ratio_out',
 'source_avg_ts_out',
 'source_sum_ts_out',
 'source_var_ts_out',
 'source_skew_ts_out',
 'source_kurtosis_ts_out',
 'source_avg_amt_out',
 'source_sum_amt_out',
 'source_var_amt_out',
 'source_skew_amt_out',
 'source_kurtosis_amt_out',
 'source_fan_in',
 'source_deg_in',
 'source_ratio_in',
 'source_avg_ts_in',
 'source_sum_ts_in',
 'source_var_ts_in',
 'source_skew_ts_in',
 'source_kurtosis_ts_in',
 'source_avg_amt_in',
 'source_sum_amt_in',
 'source_var_amt_in',
 'source_skew_amt_in',
 'source_kurtosis_amt_in',
 'dest_fan_out',
 'dest_deg_out',
 'dest_ratio_out',
 'dest_avg_ts_out',
 'dest_sum_ts_out',
 'dest_var_ts_out',
 'dest_skew_ts_out',
 'dest_kurtosis_ts_out',
 'dest_avg_amt_out',
 'dest_sum_am

In [8]:
# GFP input: one row per transaction = [txn_id, src_idx, dst_idx, seconds, amount_usd]
all_accounts   = pd.concat([edges['src_account'], edges['dst_account']]).unique()
account_to_idx = {acc: i for i, acc in enumerate(all_accounts)}

seconds = edges['Timestamp'].astype('int64') // 10**9
gfp_input = np.column_stack([
    np.arange(len(edges)),
    edges['src_account'].map(account_to_idx),
    edges['dst_account'].map(account_to_idx),
    seconds - seconds.min(),
    edges['Amount_USD'],
]).astype(np.float64)

print(f'gfp_input: {gfp_input.shape}   accounts: {len(account_to_idx):,}')

# The Windows snapml build lacks GFP, so run it inside WSL (see run_gfp_wsl.py):
# save input -> call the script -> load output
np.save('Data/_gfp_input.npy', gfp_input)
with open('Data/_gfp_params.json', 'w') as f:
    json.dump(GFP_PARAMS, f)
del gfp_input
gc.collect()

WSL_DIR = '/mnt/c/Users/User/Desktop/Sapienza/Master Thesis/gnn_for_fraudulent_patterns'
GFP_BATCH = 128   # paper setting; keeps the feature extraction causal
cmd = (f'~/gfp_env/bin/python "{WSL_DIR}/run_gfp_wsl.py" '
       f'"{WSL_DIR}/Data/_gfp_input.npy" "{WSL_DIR}/Data/_gfp_params.json" '
       f'"{WSL_DIR}/Data/_gfp_output.npy" {GFP_BATCH}')

print(f'\nRunning GFP on {len(edges):,} edges in WSL (batch size {GFP_BATCH}) ...')
with Timer('GFP via WSL'):
    result = subprocess.run(['wsl', 'bash', '-c', cmd], capture_output=True, text=True)
print(result.stdout)
assert result.returncode == 0, f'WSL GFP failed:\n{result.stderr[-2000:]}'

gfp_output = np.load('Data/_gfp_output.npy')
for tmp in ['Data/_gfp_input.npy', 'Data/_gfp_params.json', 'Data/_gfp_output.npy']:
    os.remove(tmp)

# Output = the 5 input columns passed through + the 61 features
assert gfp_output.shape == (len(edges), 5 + len(GFP_FEAT_COLS))
print(f'GFP output: {gfp_output.shape}  OK')

gfp_input: (5077237, 5)   accounts: 515,070



Running GFP on 5,077,237 edges in WSL (batch size 128) ...


  [GFP via WSL] done in 239.0s
[wsl] input (5077237, 5), streaming in batches of 128 (8 threads) ...
[wsl]   0/5,077,237 edges  (0s)
[wsl]   1,000,064/5,077,237 edges  (12s)
[wsl]   2,000,000/5,077,237 edges  (87s)
[wsl]   3,000,064/5,077,237 edges  (113s)
[wsl]   4,000,000/5,077,237 edges  (151s)
[wsl]   5,000,064/5,077,237 edges  (214s)
[wsl] done in 216.3s, output (5077237, 66)
[wsl] saved /mnt/c/Users/User/Desktop/Sapienza/Master Thesis/gnn_for_fraudulent_patterns/Data/_gfp_output.npy



GFP output: (5077237, 66)  OK


In [9]:
# Keep only the 61 feature columns (drop the 5 pass-through input columns)
gfp_feats = pd.DataFrame(gfp_output[:, 5:], columns=GFP_FEAT_COLS)
del gfp_output
gc.collect()

# Causality check on an account's FIRST outgoing transaction.
# GFP counts the current edge itself, so with perfect causality deg would be 1.
# With the paper's batch-128 streaming, an account bursting several txns inside
# the same batch (~22s of traffic) sees those few same-batch edges — that is
# the accepted, bounded exposure. Anything near the dataset scale would instead
# mean single-batch leakage (there, first-txn degrees reached 176,127).
first_txn = (edges.groupby('src_account').cumcount() == 0).to_numpy()
deg = gfp_feats.loc[first_txn, 'source_deg_out']
print(f'First-ever transactions: {first_txn.sum():,}')
print(f'  deg == 1 (perfectly causal): {(deg == 1).mean():.2%}')
print(f'  max deg: {deg.max():.0f}  (must stay below batch size {GFP_BATCH})')
assert deg.max() < GFP_BATCH, 'Degrees beyond one batch -> real future leakage!'
print('PASS — exposure bounded to a single 128-edge batch (paper protocol)')

First-ever transactions: 496,969
  deg == 1 (perfectly causal): 99.02%
  max deg: 9  (must stay below batch size 128)
PASS — exposure bounded to a single 128-edge batch (paper protocol)


## 5. Assemble Edge Feature Matrix & Save

Baseline and GFP features are concatenated into one matrix, but the two groups remain **explicitly separated** through `BASE_EDGE_COLS` / `GFP_FEAT_COLS` in `feature_meta.json`. NaNs (first transaction of an account, no history yet) are filled with 0.

In [10]:
EDGE_FEAT_COLS = BASE_EDGE_COLS + GFP_FEAT_COLS

edge_features = pd.concat([
    edges[['src_account', 'dst_account', 'label', 'Timestamp'] + BASE_EDGE_COLS],
    gfp_feats,
], axis=1)
edge_features[EDGE_FEAT_COLS] = edge_features[EDGE_FEAT_COLS].fillna(0).astype(np.float32)
del edges, gfp_feats
gc.collect()

print(f'Edge feature matrix: {edge_features.shape}')
print(f'  Baseline: {len(BASE_EDGE_COLS)}  |  GFP: {len(GFP_FEAT_COLS)}  |  total: {len(EDGE_FEAT_COLS)}')

Edge feature matrix: (5077237, 85)
  Baseline: 20  |  GFP: 61  |  total: 81


In [11]:
# Quick signal check: mean feature value for laundering vs legitimate edges.
# Ratio well above 1 means the feature separates the classes.
check = ['Amount_Log', 'Struct_Band', 'Same_Bank', 'Dt_Src_Log', 'Dt_Dst_Log',
         'Src_Bank_Risk', 'Dst_Bank_Risk',
         'scatter-gather_bins_2-3', 'temp-cycle_bins_2-3', 'lc-cycle_bins_2-3',
         'source_fan_out', 'source_deg_out', 'dest_fan_in', 'dest_deg_in',
         'source_sum_amt_out', 'dest_sum_amt_in', 'source_var_ts_out', 'dest_var_ts_in']

legit = edge_features.loc[edge_features['label'] == 0, check].mean()
laund = edge_features.loc[edge_features['label'] == 1, check].mean()
pd.DataFrame({'Legit mean': legit,
              'Laundering mean': laund,
              'Ratio': (laund / (legit + 1e-9)).round(2)}).sort_values('Ratio', ascending=False)

,Legit mean,Laundering mean,Ratio
lc-cycle_bins_2-3,1.874736e-04,3.140203e-02,167.50
temp-cycle_bins_2-3,1.874736e-04,3.140203e-02,167.50
dest_sum_amt_in,1.572520e+06,6.935578e+06,4.41
scatter-gather_bins_2-3,1.695345e-04,6.634233e-04,3.91
Struct_Band,9.228983e-03,3.029633e-02,3.28
Src_Bank_Risk,6.324645e-04,1.333479e-03,2.11
Dst_Bank_Risk,6.197297e-04,1.169952e-03,1.89
source_sum_amt_out,2.658809e+08,4.591626e+08,1.73
source_deg_out,8.514245e+02,1.385343e+03,1.63
source_fan_out,3.659187e+02,5.954447e+02,1.63


In [12]:
edge_features.to_csv('Data/edge_features.csv', index=False)
node_features.to_csv('Data/node_features.csv', index=False)

meta = {
    'BASE_EDGE_COLS': BASE_EDGE_COLS,
    'GFP_FEAT_COLS':  GFP_FEAT_COLS,
    'EDGE_FEAT_COLS': EDGE_FEAT_COLS,
    'NODE_FEAT_COLS': NODE_FEAT_COLS,
    'EDGE_DIM': len(EDGE_FEAT_COLS),
    'NODE_DIM': len(NODE_FEAT_COLS),
    'truncation': 'kept Timestamp < 2022-09-11',
    'bank_risk_smoothing_m': 200,
    'GBT_ROW': 'edge features + src node features + dst node features',
    'dropped_vs_LI_pipeline': ['Is_ACH', 'Currency_Mismatch', 'Bank_ID_Norm'],
    # ablation grid for every model notebook: which edge features to feed
    # (node features + graph structure are always used)
    'ABLATIONS': {
        'A_node_structure': [],                # no edge features at all
        'B_plus_base':      'BASE_EDGE_COLS',  # + 20 baseline edge features
        'C_plus_gfp':       'GFP_FEAT_COLS',   # + 61 GFP structural features
        'D_full':           'EDGE_FEAT_COLS',  # + all 81
    },
}
with open('Data/feature_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

vc = edge_features['label'].value_counts()
print(f'Saved edge_features.csv {edge_features.shape} and node_features.csv {node_features.shape}')
print(f'Class balance: {vc[1]:,} laundering / {vc[0]:,} legitimate ({vc[1]/vc.sum():.4%})')

Saved edge_features.csv (5077237, 85) and node_features.csv (518573, 7)
Class balance: 4,522 laundering / 5,072,715 legitimate (0.0891%)


## 6. Temporal 60/20/20 Split + Train-Fit Normalization

Edges are already timestamp-sorted; positional boundaries give train (60%) / val (20%) / test (20%).

Normalization (fit on **train only**, applied to all splits — no leakage):

| Feature group | Transformation |
|---|---|
| Pattern histograms (`*_bins_*`) | none — bounded counts |
| Vertex stats, non-negative (`fan`, `deg`, `sum`, `avg`, `var`, `kurtosis`) | log1p → clip [train p1, p99] → StandardScaler |
| Vertex stats, signed (`ratio`, `skew`) | clip [train p1, p99] → StandardScaler |
| Baseline features | none — already log-scaled / bounded / binary |

In [13]:
edge_df = edge_features   # already time-sorted; t1 / t2 defined in Section 1

for name, part in [('Train', edge_df.iloc[:t1]),
                   ('Val',   edge_df.iloc[t1:t2]),
                   ('Test',  edge_df.iloc[t2:])]:
    print(f'{name:<5}: {len(part):>9,} edges | {part["Timestamp"].min()} -> {part["Timestamp"].max()} '
          f'| laundering {part["label"].sum():,} ({part["label"].mean():.4%})')

Train: 3,046,342 edges | 2022-09-01 00:00:00 -> 2022-09-06 13:34:00 | laundering 2,297 (0.0754%)
Val  : 1,015,447 edges | 2022-09-06 13:34:00 -> 2022-09-08 16:09:00 | laundering 1,082 (0.1066%)
Test : 1,015,448 edges | 2022-09-08 16:09:00 -> 2022-09-10 23:59:00 | laundering 1,143 (0.1126%)


In [14]:
# Split the GFP columns into their three normalization groups
PATTERN_COLS = [c for c in GFP_FEAT_COLS if '_bins_' in c]                  # untouched
VERTEX_COLS  = [c for c in GFP_FEAT_COLS if c not in PATTERN_COLS]
LOG_COLS = [c for c in VERTEX_COLS if any(s in c for s in                   # non-negative
            ['fan', 'deg', 'sum', 'avg', 'var', 'kurtosis'])]
STD_COLS = [c for c in VERTEX_COLS if c not in LOG_COLS]                    # ratio, skew

print(f'untouched pattern bins: {len(PATTERN_COLS)} | log+scale: {len(LOG_COLS)} | scale-only: {len(STD_COLS)}')


def normalise(cols, use_log):
    """log1p (optional) -> clip at train p1/p99 -> standard-scale (fit on train only)."""
    vals = edge_df[cols].to_numpy()
    if use_log:
        vals = np.log1p(np.clip(vals, 0, None))
    p01 = np.percentile(vals[:t1], 1, axis=0)
    p99 = np.percentile(vals[:t1], 99, axis=0)
    vals = np.clip(vals, p01, p99)
    scaler = StandardScaler().fit(vals[:t1])
    edge_df[cols] = scaler.transform(vals).astype(np.float32)
    return {'scaler': scaler, 'p01': p01, 'p99': p99, 'cols': cols, 'log': use_log}

norm_log = normalise(LOG_COLS, use_log=True)
norm_std = normalise(STD_COLS, use_log=False)

with open('Data/standard_scaler.pkl', 'wb') as f:
    pickle.dump({'log_group': norm_log, 'std_group': norm_std,
                 'pattern_cols': PATTERN_COLS}, f)

v = edge_df[VERTEX_COLS].to_numpy()
print(f'Vertex stats after normalization: min={v.min():.2f} max={v.max():.2f} '
      f'mean={v.mean():.3f} std={v.std():.3f}')
print('Scalers saved -> Data/standard_scaler.pkl')

untouched pattern bins: 9 | log+scale: 40 | scale-only: 12


Vertex stats after normalization: min=-9.57 max=5.89 mean=0.027 std=1.005
Scalers saved -> Data/standard_scaler.pkl


## 7. Graph Construction — Cumulative PyG Snapshots

- **train_graph**: train edges only, all evaluated
- **val_graph**: train + val edges; only the val portion has `eval_mask=True`
- **test_graph**: all edges; only the test portion has `eval_mask=True`

Context edges get label `-1` and are excluded from loss/metrics. This gives the GNN full
graph context at inference time while keeping evaluation leakage-free. Snapshots are
built, saved and freed **one at a time** to stay inside local RAM.

In [15]:
# Node feature matrix: one row per account, ordered by account_to_idx
X_node = torch.tensor(
    node_features.set_index('account_id')
                 .reindex(account_to_idx.keys())     # align rows to graph node ids
                 [NODE_FEAT_COLS]
                 .fillna(0)                          # accounts missing from accounts file
                 .to_numpy(dtype=np.float32)
)
N_nodes = len(account_to_idx)
print(f'Nodes: {N_nodes:,}   node features: {tuple(X_node.shape)}')


def save_snapshot(name, n_rows, eval_start):
    """Build a PyG graph from the first n_rows edges and save it.

    Edges before eval_start are context only (label -1, excluded from loss);
    edges from eval_start on are the ones this split evaluates.
    """
    part = edge_df.iloc[:n_rows]
    eval_mask = np.arange(n_rows) >= eval_start

    labels = np.where(eval_mask, part['label'].to_numpy(), -1)
    graph = Data(
        x          = X_node,
        edge_index = torch.tensor(np.stack([
                         part['src_account'].map(account_to_idx).to_numpy(),
                         part['dst_account'].map(account_to_idx).to_numpy()])),
        edge_attr  = torch.tensor(part[EDGE_FEAT_COLS].to_numpy(dtype=np.float32)),
        edge_time  = torch.tensor(part['Timestamp'].astype('int64').to_numpy() // 10**9),
        y          = torch.tensor(labels),
        eval_mask  = torch.tensor(eval_mask),
        num_nodes  = N_nodes,
    )
    n_eval  = int(eval_mask.sum())
    n_laund = int((labels == 1).sum())
    print(f'{name:<12}| edges={n_rows:>9,} | eval={n_eval:>9,} '
          f'| laundering={n_laund:,} ({n_laund/n_eval:.4%})')
    torch.save(graph, f'Data/{name}.pt')
    del graph
    gc.collect()


with Timer('train_graph'):
    save_snapshot('train_graph', n_rows=t1,      eval_start=0)
with Timer('val_graph'):
    save_snapshot('val_graph',   n_rows=t2,      eval_start=t1)
with Timer('test_graph'):
    save_snapshot('test_graph',  n_rows=n_edges, eval_start=t2)

with open('Data/account_to_idx.pkl', 'wb') as f:
    pickle.dump(account_to_idx, f)

# Reload check
g = torch.load('Data/train_graph.pt', weights_only=False)
assert g.edge_attr.shape[1] == len(EDGE_FEAT_COLS)
print(f'\nReload check OK — edge_attr {tuple(g.edge_attr.shape)}')
del g

print()
print('=' * 55)
print(f'  EDGE_DIM = {len(EDGE_FEAT_COLS)}   (base {len(BASE_EDGE_COLS)} + gfp {len(GFP_FEAT_COLS)})')
print(f'  NODE_DIM = {len(NODE_FEAT_COLS)}')
print('=' * 55)

Nodes: 515,070   node features: (515070, 6)


train_graph | edges=3,046,342 | eval=3,046,342 | laundering=2,297 (0.0754%)


  [train_graph] done in 3.9s


val_graph   | edges=4,061,789 | eval=1,015,447 | laundering=1,082 (0.1066%)


  [val_graph] done in 5.5s


test_graph  | edges=5,077,237 | eval=1,015,448 | laundering=1,143 (0.1126%)


  [test_graph] done in 7.9s



Reload check OK — edge_attr (3046342, 81)

  EDGE_DIM = 81   (base 20 + gfp 61)
  NODE_DIM = 6


## Appendix — GFP Feature vs Label Correlation

GFP features should show measurable correlation with the laundering label; uniformly weak
correlation would signal a configuration problem.

In [16]:
corr = (edge_df[GFP_FEAT_COLS + ['label']].corr()['label']
        .drop('label').sort_values(ascending=False))
print('Top 10 GFP features by correlation with the label:')
print(corr.head(10).round(4).to_string())
print('\nBottom 5 (negative):')
print(corr.tail(5).round(4).to_string())
print(f'\nFeatures with |corr| > 0.01: {(corr.abs() > 0.01).sum()} / {len(corr)}')

Top 10 GFP features by correlation with the label:
temp-cycle_bins_2-3      0.0635
lc-cycle_bins_2-3        0.0635
lc-cycle_bins_3-5        0.0589
temp-cycle_bins_3-5      0.0434
lc-cycle_bins_5-inf      0.0285
dest_fan_in              0.0202
temp-cycle_bins_5-inf    0.0172
source_avg_ts_out        0.0114
dest_avg_ts_in           0.0114
source_avg_amt_out       0.0104

Bottom 5 (negative):
source_kurtosis_ts_out      -0.0119
source_var_ts_out           -0.0127
dest_ratio_in               -0.0177
source_ratio_out            -0.0202
scatter-gather_bins_5-inf       NaN

Features with |corr| > 0.01: 14 / 61
